# EDA: CICIDS Infiltration + synthetic lateral-movement rows

## Label inventory (combined 8 CSVs)

Ran `value_counts()` on the combined CICIDS2017 label column. Closest match to lateral movement is **Infiltration** (36 rows) — CICIDS docs describe post-compromise internal Nmap from a compromised host, not just the name sounding close.

## Synthetic generation (`src/data/synthesize_infiltration.py`)

- Approach: row-wise template + ±15% numeric jitter, clamped to real min/max.
- Per-template IP cache: same template → same synthetic source/dest IP (repeat-offender host for SQL correlation); different templates never share an IP.
- Verified at `n_rows=200`: 36 unique synthetic source IPs (= 36 templates), consistent repeats, zero cross-template / real IP collisions.

### Limitation — port/protocol diversity

Synthetic rows vary **source/dest host** and **traffic volume** (duration, packet counts, sizes), **not** port/protocol patterns. Ports and protocol are copied straight from the 36 real templates, so synthetic Infiltration traffic is capped at whatever port/protocol variety those 36 rows already contain. Not a bug — a consequence of the small real-sample size and the plausibility constraint (don't invent service combos that never co-occurred). Worth remembering if anyone asks why synthetic flows cluster on the same handful of ports (e.g. dest 444).